# 1. Import Libraries and CSV
Following our ARIMA models and autoarima EDA, we're going to want to confirm many of autoarima's findings for academic rigor and validity manually.

In [ ]:
# Import our custom utility module
import dc_housing_time_series_utils as ts_utils

# Import standard libraries
import pandas as pd
import warnings
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
import statsmodels.tsa.vector_ar.vecm as vecm
from statsmodels.tsa.api import VAR
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Load data
df = pd.read_csv("DC_Master_Missing.csv", parse_dates=['Date'], index_col='Date')

print(df.tail())

# 2. Sort Columns Again

In [ ]:
df = pd.read_csv('DC_Master_Missing.csv', index_col=0, parse_dates=True) 

# Calculate missing values per column
missing_counts = df.isnull().sum()

# Sort columns by missing count (least to most)
print("\nSorting columns")
sorted_columns = missing_counts.sort_values().index.tolist()
print("Pandas Sort Order:", sorted_columns)

# 3. Run KPSS and Stationarity Tests
Let's recall our previous results from autoarima, and let's see if the results are the same for KPSS and ADF.

Unemployment_Rate: ARIMA(2,0,1)(1,0,0)[4] -> d=0, D=0 (Suggests original series is stationary)

Interest_Rate: ARIMA(2,1,2)(0,0,0)[4] -> d=1, D=0 (Suggests 1st order differencing needed)

Mortgage_Rate: ARIMA(0,1,0)(0,0,0)[4] -> d=1, D=0 (Suggests 1st order differencing needed)

CPI: ARIMA(1,1,0)(1,0,1)[4] -> d=1, D=0 (Suggests 1st order differencing needed)

House_Index: ARIMA(0,1,3)(1,0,0)[4] -> d=1, D=0 (Suggests 1st order differencing needed)

GDP: ARIMA(0,2,1)(0,0,0)[4] -> d=2, D=0 (Suggests 2nd order differencing needed)

Population: ARIMA(1,1,0)(0,0,1)[4] -> d=1, D=0 (Suggests 1st order differencing needed)

Poverty_Rate: ARIMA(2,1,0)(1,0,2)[4] -> d=1, D=0 (Suggests 1st order differencing needed)

Median_Household_Income: ARIMA(1,1,3)(0,0,0)[4] -> d=1, D=0 (Suggests 1st order differencing needed)

For loop to run kpss and stationarity tests on each column

In [ ]:
for col_name in sorted_columns:
    print(f"\nProcessing column: {col_name}")
    ts_utils.test_stationarity(df[col_name], col_name)

Unemployment rate is already stationary! No differencing required there.

First, create differenced columns

In [ ]:
df = ts_utils.create_differenced_columns(
    df, 
    ['Interest_Rate', 'Mortgage_Rate', 'CPI', 'House_Index', 'GDP', 'Population', 'Poverty_Rate', 'Median_Household_Income']
)

Run differenced columns through the stationarity tests

In [ ]:
for col_name in ['Interest_Rate_diff1', 'Mortgage_Rate_diff1', 'CPI_diff1', 'House_Index_diff1', 'GDP_diff1', 'Population_diff1', 'Poverty_Rate_diff1', 'Median_Household_Income_diff1']:
    print(f"\nProcessing column: {col_name}")
    ts_utils.test_stationarity(df[col_name], col_name)


Interest Rate, Mortgage Rate, CPI, House Index, Poverty Rate, and Median Household income are stationary at integrated of order 1, let's see what we find with another difference on GDP and population.

In [ ]:

# Create 2nd differenced columns for GDP and Population
df = ts_utils.create_differenced_columns(
    df, 
    ['GDP_diff1', 'Population_diff1'],
    diff_order=1,  # This creates GDP_diff1_diff1 = GDP_diff2 and Population_diff1_diff1 = Population_diff2
    inplace=True
)


In [ ]:
# Ensure 2nd differenced columns are created properly
for col_name in ['GDP_diff1', 'Population_diff1']:
    diff_col2 = f'{col_name}_diff1'  # Creates GDP_diff1_diff1 and Population_diff1_diff1
    df[col_name.replace('diff1', 'diff2')] = df[diff_col2]  # Rename to GDP_diff2 and Population_diff2

# Run 2nd differenced columns through the stationarity tests
for col_name in ['GDP_diff2', 'Population_diff2']:
    print(f"\nProcessing column: {col_name}")
    ts_utils.test_stationarity(df[col_name], col_name)

Stationarity was confirmed using the ADF test on the second-differenced series (p < 0.05). Although the KPSS test marginally rejected the null of stationarity (p ≈ 0.0417), the evidence was not strong, and AutoARIMA modeling supported second differencing (d = 2). We'll just go with l2. We also want to note that autoarima chose l1 for population, which makes sense considering that it marginally fails the ADF fuller test at 0.812 at the first difference.

# 4. Seasonal Decomposition Visualization
Since D = 0 from autoarima, it did not need to do differencing for seasonal components, but it was P = 1 for 3, meaning there's a seasonal component it adjusted for based on the previous year's season, but not between seasons. Let's visualize it.

In [ ]:
for var in ['Unemployment_Rate', 'CPI', 'Poverty_Rate']:
    ts_utils.plot_seasonal_decomposition(df, var)

Let's make the differenced dataframes with what we just found on our ARIMA filled CSV.

In [ ]:
# First, create differenced columns
df = pd.read_csv("DC_Master_ARIMA_Filled.csv", parse_dates=['Date'], index_col='Date')
# Create differenced columns for all variables except Unemployment_Rate
df = ts_utils.create_differenced_columns(
    df,
    ['Interest_Rate', 'Mortgage_Rate', 'Population', 'CPI', 'House_Index', 'GDP', 'Population', 'Poverty_Rate', 'Median_Household_Income']
)
# Ensure 2nd differenced GDP is created
df['GDP_diff2'] = df['GDP_diff1'].diff()
# Drop rows with NaN values
df.dropna(inplace=True)
# Save the DataFrame to a new CSV file
df.to_csv("DC_Master_Stationary.csv", index=True)

# 5. Johansen Test
Based on my research for forecasting methods, VAR and VECM are quite popular. In order to do them, we need to perform the Johansen test on the same integrated order of the variables.

In [ ]:
try:
    # Load the dataset with stationary transformations
    df = pd.read_csv("DC_Master_Stationary.csv", parse_dates=['Date'], index_col='Date')

    # Optional: Filter dates if needed
    df = df[df.index <= '2024-10-01']

    print(f"Data loaded successfully. Shape: {df.shape}")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"Columns: {df.columns.tolist()}")

    # Define variables by integration order based on previous analysis
    vars_i1 = ['Interest_Rate', 'Mortgage_Rate', 'CPI', 'House_Index',
               'Population', 'Poverty_Rate', 'Median_Household_Income']
    vars_i2 = ['GDP']

    # Perform Johansen test
    print(f"\n--- Performing Johansen Cointegration Test ---")
    print(f"Variables included: {vars_i1 + ['GDP_diff1']}")
    
    # Run the Johansen test
    k_ar_diff_selected, result = ts_utils.run_johansen_test(
        df, 
        vars_i1=vars_i1, 
        vars_i2=vars_i2,
        max_lags=8, 
        det_order=0
    )
    
    print(f"\nSelected lag order k_ar_diff = {k_ar_diff_selected}")
    
    # Display and interpret the test results
    cointegration_rank = ts_utils.display_johansen_results(result)

except FileNotFoundError:
    print(f"Error: The file 'DC_Master_Stationary.csv' was not found. Please ensure it's in the correct directory.")
except ValueError as ve:
    print(ve)
except Exception as e:
    print(f"An unexpected error occurred: {e}")


The Johansen cointegration test was conducted using variables integrated of order one (I(1)) and the first difference of GDP (treated as I(2)), with a lag order k_ar_diff = 1 selected based on BIC and HQIC criteria. At the 95% significance level, the test results showed some divergence: the Max-Eigenvalue statistic indicated a cointegrating rank of r = 1, while the Trace statistic suggested r <= 2. Following the common practice of relying on the Max-Eigenvalue test in case of divergence, a rank of r = 1 was selected. This provides evidence supporting one stable long-run equilibrium relationship among the variables, confirming that a Vector Error Correction Model (VECM) is appropriate for modeling both these long-run dynamics and the system's short-run adjustments. We also got ranks of 6 with a lag of 8 using AIC, and we tried them as well.

# 6. VECM Modeling Attempts

In [ ]:

try:
    # Load the dataset
    df = pd.read_csv("DC_Master_Stationary.csv", parse_dates=['Date'], index_col='Date')

    # Optional: Filter dates
    df = df[df.index <= '2024-10-01']

    print(f"Data loaded successfully. Shape: {df.shape}")

    # Define variables by integration order
    vars_i1 = ['Interest_Rate', 'Mortgage_Rate', 'CPI', 'House_Index',
                'Population', 'Poverty_Rate', 'Median_Household_Income']
    vars_i2 = ['GDP']

    # Prepare data for VECM
    df['GDP_diff1'] = df[vars_i2[0]].diff() if 'GDP_diff1' not in df.columns else df['GDP_diff1']
    vars_for_vecm = vars_i1 + ['GDP_diff1']
    df_vecm_input = df[vars_for_vecm].copy()

    # Handle missing values
    df_vecm_dropna = df_vecm_input.dropna()

    print(f"\n--- Data Preparation for VECM ---")
    print(f"Variables included:")
    print(df_vecm_dropna.columns.tolist())
    print(f"Data shape for VECM after dropping NaNs: {df_vecm_dropna.shape}")

    # Define VECM parameters 
    lag_order_k_ar_diff = 1  # From Johansen test BIC/HQIC
    cointegration_rank_r = 2  # Based on results from previous section
    deterministic_term = 'co'  # Constant in cointegrating equation only

    print(f"\n--- VECM Specification ---")
    print(f"Endogenous Variables: {df_vecm_dropna.shape[1]}")
    print(f"Lag Order (k_ar_diff): {lag_order_k_ar_diff}")
    print(f"Cointegrating Rank (r): {cointegration_rank_r}")
    print(f"Deterministic Term: '{deterministic_term}'")

    # Fit the VECM model
    vecm_results = ts_utils.fit_vecm_model(
        df_vecm_dropna,
        variables=vars_for_vecm,
        cointegration_rank=cointegration_rank_r,
        lag_order=lag_order_k_ar_diff,
        deterministic_term=deterministic_term
    )

    # Display VECM results summary
    print("\n--- VECM Estimation Results ---")
    print(vecm_results.summary())

    # Residual diagnostics
    print("\n--- Residual Diagnostics ---")
    print("\nResidual Autocorrelation Test (Whiteness Test):")
    try:
        # Test up to a certain number of lags
        max_diag_lags = min(8, (len(df_vecm_dropna) - lag_order_k_ar_diff * len(vars_for_vecm)) // 2)
        if max_diag_lags > 0:
            print(vecm_results.test_whiteness(nlags=max_diag_lags, adjusted=True).summary())
        else:
            print("Not enough data for residual autocorrelation test.")
    except Exception as diag_e:
        print(f"Could not perform residual autocorrelation test: {diag_e}")

    print("\nResidual Normality Test:")
    try:
        print(vecm_results.test_normality().summary())
    except Exception as diag_e:
        print(f"Could not perform residual normality test: {diag_e}")

    # Plot the residuals
    print("\n--- Plotting VECM Residuals ---")
    ts_utils.plot_residuals(vecm_results, title='VECM Residuals Over Time')

except FileNotFoundError:
    print(f"Error: The file 'DC_Master_Stationary.csv' was not found.")
except ValueError as ve:
    print(ve)
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Now let's try a VECM with exogenous variables to account for structural breaks

In [ ]:
try:
    # Load the dataset
    df = pd.read_csv("DC_Master_Stationary.csv", parse_dates=['Date'], index_col='Date')

    # Filter dates
    df = df[df.index <= '2024-10-01']

    print(f"Data loaded successfully. Shape: {df.shape}")

    # Create dummy variables for crisis and COVID periods
    crisis_periods = {
        'dummy_crisis_2008_period': ('2007-10-01', '2009-10-01'),
        'dummy_covid_2020_period': ('2020-01-01', '2021-10-01')
    }
    df = ts_utils.create_dummies(df, crisis_periods)

    # Define variables for VECM
    vars_i1 = ['Interest_Rate', 'Mortgage_Rate', 'CPI', 'House_Index',
               'Population', 'Poverty_Rate', 'Median_Household_Income']
    vars_i2 = ['GDP']
    exog_cols = ['dummy_crisis_2008_period', 'dummy_covid_2020_period', 'Unemployment_Rate']

    # Prepare endogenous data
    df['GDP_diff1'] = df[vars_i2[0]].diff() if 'GDP_diff1' not in df.columns else df['GDP_diff1']
    vars_for_vecm = vars_i1 + ['GDP_diff1']
    df_vecm_endog = df[vars_for_vecm].copy()

    # Prepare exogenous data
    df_exog = df[exog_cols].copy()

    # Align indices and handle missing values
    # Combine aligned endog and exog, drop rows with any NaNs, then split
    temp_combined = pd.concat([df_vecm_endog, df_exog], axis=1)
    temp_combined_dropna = temp_combined.dropna()
    df_vecm_final_endog = temp_combined_dropna[df_vecm_endog.columns]
    df_exog_final = temp_combined_dropna[df_exog.columns]

    print(f"\n--- Final Data Shapes for VECM with Exogenous Variables ---")
    print(f"Endogenous data shape: {df_vecm_final_endog.shape}")
    print(f"Exogenous data shape: {df_exog_final.shape}")

    # VECM parameters
    lag_order_k_ar_diff = 1
    cointegration_rank_r = 1
    deterministic_term = 'co'

    print(f"\n--- VECM Specification with Exogenous Variables ---")
    print(f"Endogenous Variables: {df_vecm_final_endog.shape[1]}")
    print(f"Exogenous Variables: {df_exog_final.shape[1]} {df_exog_final.columns.tolist()}")
    print(f"Lag Order (k_ar_diff): {lag_order_k_ar_diff}")
    print(f"Cointegrating Rank (r): {cointegration_rank_r}")
    print(f"Deterministic Term: '{deterministic_term}'")

    # Fit the VECM model with exogenous variables
    vecm_results_exog = ts_utils.fit_vecm_model(
        df_vecm_final_endog,
        variables=df_vecm_final_endog.columns.tolist(),  # Use all columns
        cointegration_rank=cointegration_rank_r,
        lag_order=lag_order_k_ar_diff,
        deterministic_term=deterministic_term,
        exog=df_exog_final
    )

    # Display VECM results summary
    print("\n--- VECM Estimation Results (with Exogenous Variables) ---")
    print(vecm_results_exog.summary())

    # Residual diagnostics
    print("\n--- Residual Diagnostics (with Exogenous Variables) ---")
    
    # Whiteness test
    print("\nResidual Autocorrelation Test (Whiteness Test):")
    try:
        max_diag_lags = min(8, (len(df_vecm_final_endog) - lag_order_k_ar_diff * df_vecm_final_endog.shape[1]) // 2)
        if max_diag_lags > 0:
            print(vecm_results_exog.test_whiteness(nlags=max_diag_lags, adjusted=True).summary())
        else:
            print("Not enough data points for residual autocorrelation test.")
    except Exception as diag_e:
        print(f"Could not perform residual autocorrelation test: {diag_e}")

    # Normality test
    print("\nResidual Normality Test:")
    try:
        print(vecm_results_exog.test_normality().summary())
    except Exception as diag_e:
        print(f"Could not perform residual normality test: {diag_e}")

    # Plot residuals
    print("\n--- Plotting VECM Residuals (with Exogenous Variables) ---")
    ts_utils.plot_residuals(vecm_results_exog, title='VECM Residuals Over Time (with Exogenous Variables)')

except Exception as e:
    print(f"An error occurred: {e}")

# 7. VECM Summary and Conclusions
Initial analysis involved testing each time series for stationarity, leading to differencing variables to achieve I(0) status (Unemployment Rate I(0), GDP/Population I(2), others I(1)). A Johansen cointegration test performed on the appropriately transformed I(1) variables indicated the presence of one cointegrating relationship (rank=1). This suggested that a Vector Error Correction Model (VECM) would be the theoretically appropriate approach to capture both short-run dynamics and the long-run equilibrium. VECM models were subsequently estimated using lag orders suggested by information criteria (k=1 via BIC/HQIC, and k=7 via AIC/FPE).
Unfortunately, residual diagnostic tests for both models revealed persistent and significant autocorrelation, indicating model misspecification. Given these results, the VECM framework did not adequately capture the dynamics within this specific system. Therefore, we will pivot to exploring alternative modeling techniques
Let's change everything to a percentage change, then test for stationarity. We can try out VAR.

In [ ]:
# Create percentage change dataframe
df = pd.read_csv("DC_Master_ARIMA_Filled.csv", parse_dates=['Date'], index_col='Date')
df = ts_utils.create_pct_change_columns(df, exclude_cols=['Unemployment_Rate'])

# Drop NANs
df.dropna(inplace=True)

# Save the DataFrame to a new CSV file
df.to_csv("DC_Master_ARIMA_Filled_pct_change.csv", index=True)

# Load the percentage change data and test for stationarity
df = pd.read_csv("DC_Master_ARIMA_Filled_pct_change.csv", parse_dates=['Date'], index_col='Date')
df = df[df.index <= '2024-10-01']

# Test stationarity of percentage change columns
for col_name in df.columns:
    if 'pct_change' in col_name:
        print(f"\nProcessing column: {col_name}")
        ts_utils.test_stationarity(df[col_name], col_name)
    else:
        print(f"Skipping non-pct_change column: {col_name}")

Using percentage changes (pct_change) transforms the variables differently than differencing (diff). If most variables are now stationary after the first percentage change, it implies the original levels (or rather, their logarithms) were likely I(1). For the population pct change, we can account for this by establishing it as a constant trend.

# VAR Analysis

In [ ]:
try:
    # Load percentage change data
    df_pct = pd.read_csv("DC_Master_ARIMA_Filled_pct_change.csv", parse_dates=['Date'], index_col='Date')
    df_pct = df_pct[df_pct.index <= '2024-10-01']
    print(f"Data loaded successfully. Shape: {df_pct.shape}")

    # Select variables for VAR
    var_cols = ['Interest_Rate_pct_change', 'Mortgage_Rate_pct_change', 'CPI_pct_change',
                'House_Index_pct_change', 'Population_pct_change', 'Poverty_Rate_pct_change',
                'Median_Household_Income_pct_change', 'GDP_pct_change', 'Unemployment_Rate']

    print(f"\nSelecting columns for VAR: {var_cols}")
    df_var_data = df_pct[var_cols].copy()

    # Fit VAR model with lag selection
    model, var_results, selected_lag = ts_utils.fit_var_model(
        df_var_data,
        variables=var_cols,
        lag_order=None,  # Will use information criteria
        max_lags=8,
        trend='ct'  # Constant and trend
    )

    # Display VAR results
    print("\n--- VAR Estimation Results ---")
    print(var_results.summary())

    # Residual diagnostics
    print("\n--- Residual Diagnostics ---")
    
    # Whiteness test (autocorrelation)
    print("\nResidual Autocorrelation Test (Whiteness Test):")
    try:
        # Try test_whiteness as the method for Portmanteau test
        portmanteau_results = var_results.test_whiteness(nlags=8, adjusted=True)
        print(portmanteau_results.summary())
    except AttributeError:
        print("'.test_whiteness()' not found. Trying alternative methods...")
        from statsmodels.stats.diagnostic import acorr_ljungbox
        
        # Calculate manually for each residual series
        for i, col in enumerate(df_var_data.columns):
            lb_test = acorr_ljungbox(var_results.resid[:,i], lags=[8], return_df=True)
            p_val = lb_test['lb_pvalue'].iloc[0]
            print(f"Ljung-Box test for {col}: p-value = {p_val:.4f}")
    except Exception as diag_e:
        print(f"Could not perform residual autocorrelation test: {diag_e}")

    # Normality test
    print("\nResidual Normality Test:")
    try:
        normality_results = var_results.test_normality()
        print(normality_results.summary())
    except Exception as diag_e:
        print(f"Could not perform residual normality test: {diag_e}")

    # Plot residuals
    print("\n--- Plotting VAR Residuals ---")
    ts_utils.plot_residuals(var_results, title='VAR Model Residuals')

    # Forecast
    print("\n--- Forecasting ---")
    num_forecast_steps = 8
    print(f"Generating forecast for the next {num_forecast_steps} steps...")
    
    # Get the last observations needed for forecasting (depends on lag order)
    last_observations = df_var_data.values[-selected_lag:]
    
    # Forecast
    forecast_values = var_results.forecast(y=last_observations, steps=num_forecast_steps)
    
    # Create forecast index (next quarters)
    # If frequency not detected, assume quarterly
    freq = df_var_data.index.freq
    if freq is None:
        print("Warning: Data frequency not detected, assuming quarterly for forecast index.")
        freq = pd.tseries.offsets.QuarterEnd()
        
    forecast_index = pd.date_range(
        start=df_var_data.index[-1] + freq,
        periods=num_forecast_steps,
        freq=freq
    )
    
    # Create forecast DataFrame
    forecast_df = pd.DataFrame(
        forecast_values, 
        index=forecast_index, 
        columns=[f'{col}_forecast' for col in df_var_data.columns]
    )
    
    print(f"\nForecast for the next {num_forecast_steps} steps:")
    print(forecast_df)

except Exception as e:
    print(f"An error occurred: {e}")

Create crisis and COVID dummies and save to CSV for future use

In [ ]:
df = pd.read_csv("DC_Master_ARIMA_Filled_pct_change.csv", parse_dates=['Date'], index_col='Date')

# Create crisis and COVID period dummies
crisis_periods = {
    'dummy_crisis_2008_period': ('2007-10-01', '2009-10-01'),
    'dummy_covid_2020_period': ('2020-01-01', '2021-10-01')
}
df = ts_utils.create_dummies(df, crisis_periods)

# Save the DataFrame to a new CSV file
df.to_csv("DC_Master_ARIMA_Filled_pct_change_dummies.csv", index=True)

One more thing, let's plot ACF and PACF of House_Index

In [ ]:
df = df[df.index <= '2024-10-01']
# Select the column of interest
col_name = 'House_Index_pct_change'
series = df[col_name].dropna().astype(float)  # Drop NaNs and ensure float type
# Plot ACF and PACF
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
plot_acf(series, lags=12, ax=axes[0], title=f'ACF of {col_name}')
plot_pacf(series, lags=12, ax=axes[1], title=f'PACF of {col_name}')